In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# El aviso de «pocos clientes de calibración» es una red de seguridad: no se silencia
warnings.filterwarnings("default", message=".*clientes de calibración.*")
# Permite importar el paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 05 · Torneo de modelos — descartar con números, no coronar

Especificación: `especificaciones/05-torneo.md` · Plan: `specs/003-model-tournament/plan.md`.
Este notebook **solo orquesta**: llama a funciones de `src/olistbr/` y muestra resultados.

**Qué se decide aquí.** El torneo no busca un campeón: busca **1 o 2 finalistas** para optimizar en la etapa siguiente.
Todos los candidatos usan **los mismos 5 pliegues por cliente** de la etapa 4 y la misma **prueba hacia adelante**
(entrenar hasta 2018-02-28, validar de marzo a agosto de 2018).

**Qué gana.** El **costo 3 a 1** del plazo recomendado: subestimar cuesta 3 veces más que sobrestimar, lo que equivale a 4 veces la
pérdida cuantílica de 0,75; por eso el plazo óptimo es el **percentil 75**. El MAE, el RMSE y el R² se reportan siempre.

**Dos familias de modelos.**
- **A (valor típico + margen):** el modelo predice un valor y se le suma un margen = percentil 75 de los errores en **clientes de
  calibración** apartados dentro de cada entrenamiento (nunca de la validación).
- **B (percentil 75 directo):** el modelo ya predice el plazo; no lleva margen.

⚠️ El MAE, RMSE y R² de la familia B **no son comparables** con los de la A (su predicción es el percentil 75, más alto a propósito).
Entre familias se compara **solo el costo**.

**Reglas de esta etapa:** solo se usa el conjunto de entrenamiento (la prueba final queda cerrada); sin la marca de fin de año ni el
historial por vendedor; el plazo prometido por Olist nunca entra.

In [2]:
import pandas as pd

from olistbr.config import FORWARD_TRAIN_END, FORWARD_VALID_END, FORWARD_VALID_START, N_FOLDS, PROCESSED_DIR, REPORTS_DIR, TARGET, TOURNAMENT_GROUPS
from olistbr.data.split import forward_split, make_group_folds
from olistbr.features.engineering import add_features
from olistbr.models.evaluation import (
    compare_models, coverage_by_month, evaluate_candidate_folds, evaluate_candidate_holdout, per_thousand, regional_metrics, tournament_table,
)
from olistbr.models.train import candidates

pd.set_option("display.width", 230)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

BASELINES = {"mediana": "Línea base: siempre la mediana", "plazo_fijo": "Línea base: plazo fijo (percentil 75)"}
assert not {"peak_season", "history"} & set(TOURNAMENT_GROUPS), "el torneo no usa la marca de fin de año ni el historial"
print("grupos de variables del torneo:", ", ".join(TOURNAMENT_GROUPS))

grupos de variables del torneo: logs, same_state, regions, calendar, distance_interaction, ratios, multi_seller


## 1. Datos, pliegues y prueba hacia adelante

Se leen los datos de entrenamiento, se calculan las variables derivadas **una vez** y se crean los pliegues **una vez**.

In [3]:
train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
df = add_features(train)
X, y = df.drop(columns=[TARGET]), df[TARGET]

folds = make_group_folds(df)          # verifica que ningún cliente cae en los dos lados de un pliegue
assert sorted(i for _, valid_idx in folds for i in valid_idx) == list(range(len(df))), "los pliegues no cubren cada fila una vez"
fwd_train, fwd_valid = forward_split(df)

print(f"entrenamiento: {len(df):,} pedidos · {df['customer_unique_id'].nunique():,} clientes · {N_FOLDS} pliegues de {[len(v) for _, v in folds]}")
print(f"hacia adelante: entrenamiento {len(fwd_train):,} (hasta {FORWARD_TRAIN_END}) · validación {len(fwd_valid):,} ({FORWARD_VALID_START} a {FORWARD_VALID_END})")
print("percentil 75 de los días en entrenamiento (el «mejor plazo fijo»):", round(y.quantile(0.75), 2))

entrenamiento: 76,924 pedidos · 74,425 clientes · 5 pliegues de [15353, 15353, 15401, 15439, 15378]
hacia adelante: entrenamiento 44,850 (hasta 2018-02-28) · validación 31,447 (2018-03-01 a 2018-08-31)
percentil 75 de los días en entrenamiento (el «mejor plazo fijo»): 15.7


## 2. Correr el torneo

Cada candidato se ajusta en los 5 pliegues y en la partición hacia adelante. Se guarda **cada pliegue**, no solo el promedio.

In [4]:
frames, forward_rows = [], []
for name, candidate in candidates().items():
    frames.append(evaluate_candidate_folds(name, candidate, X, y, folds))
    forward_rows.append(evaluate_candidate_holdout(name, candidate, fwd_train, fwd_valid))
    mean_fit = frames[-1]["fit_seconds"].mean()
    print(f"{name:42s} familia {candidate.family:4s} listo  (ajuste medio {mean_fit:5.1f} s)")

by_fold = pd.concat(frames, ignore_index=True)
forward = pd.DataFrame(forward_rows)
len(by_fold), len(forward)

Línea base: siempre la mediana             familia base listo  (ajuste medio   0.4 s)


Línea base: plazo fijo (percentil 75)      familia base listo  (ajuste medio   0.4 s)


Regresión lineal                           familia A    listo  (ajuste medio   0.6 s)


Ridge                                      familia A    listo  (ajuste medio   0.5 s)


Bosque aleatorio liviano                   familia A    listo  (ajuste medio   6.2 s)


LightGBM                                   familia A    listo  (ajuste medio   2.1 s)


XGBoost                                    familia A    listo  (ajuste medio   3.2 s)


LightGBM cuantil 0,75                      familia B    listo  (ajuste medio   2.6 s)


XGBoost cuantil 0,75                       familia B    listo  (ajuste medio   3.2 s)


HistGradientBoosting cuantil 0,75          familia B    listo  (ajuste medio  11.7 s)


(50, 10)

## 3. Resultados: la tabla del torneo

Ordenada por **costo 3 a 1** (media entre los 5 pliegues; menor es mejor). `supera_*` dice si el modelo vence a cada línea base por más
que su ruido; `empate_con_el_mejor` marca los que no se distinguen del mejor.

In [5]:
table = tournament_table(by_fold, forward, BASELINES)
business = table[["modelo", "familia", "costo_media", "costo_desv", "adelante_costo", "tarde_media", "cobertura_media", "margen_media",
                  "posicion_costo", "supera_mediana", "supera_plazo_fijo", "mejor", "empate_con_el_mejor"]]
business.round(3)

,modelo,familia,costo_media,costo_desv,adelante_costo,tarde_media,cobertura_media,margen_media,posicion_costo,supera_mediana,supera_plazo_fijo,mejor,empate_con_el_mejor
0,"LightGBM cuantil 0,75",B,9.367,0.040,9.230,0.256,0.744,0.000,1,True,True,True,False
1,"HistGradientBoosting cuantil 0,75",B,9.400,0.049,9.257,0.254,0.746,0.000,2,True,True,False,False
2,LightGBM,A,9.470,0.033,9.373,0.247,0.753,2.263,3,True,True,False,False
3,XGBoost,A,9.471,0.036,9.416,0.247,0.753,2.277,4,True,True,False,False
4,"XGBoost cuantil 0,75",B,9.528,0.054,9.307,0.251,0.749,0.000,5,True,True,False,False
5,Bosque aleatorio liviano,A,9.612,0.055,9.387,0.248,0.752,2.300,6,True,True,False,False
6,Regresión lineal,A,9.664,0.044,9.433,0.247,0.753,2.395,7,True,True,False,False
7,Ridge,A,9.664,0.044,9.434,0.247,0.753,2.395,8,True,True,False,False
8,Línea base: plazo fijo (percentil 75),base,11.840,0.087,11.804,0.250,0.750,0.000,9,False,False,False,False
9,Línea base: siempre la mediana,base,14.334,0.077,11.697,0.500,0.500,0.000,10,False,False,False,False


## 4. Exactitud: MAE, RMSE, R² y sesgo

Se comparan **dentro de cada familia**. La familia B predice el percentil 75, así que su MAE y su R² salen peores a propósito.

In [6]:
accuracy = table[["modelo", "familia", "mae_media", "mae_desv", "rmse_media", "r2_media", "sesgo_media", "adelante_mae",
                  "posicion_mae", "posicion_costo"]]
accuracy.sort_values(["familia", "mae_media"]).round(3)

,modelo,familia,mae_media,mae_desv,rmse_media,r2_media,sesgo_media,adelante_mae,posicion_mae,posicion_costo
2,LightGBM,A,4.981,0.010,7.491,0.290,0.002,5.506,1,3
3,XGBoost,A,4.990,0.013,7.508,0.287,-0.015,5.551,2,4
5,Bosque aleatorio liviano,A,5.055,0.014,7.570,0.275,-0.001,5.463,3,6
6,Regresión lineal,A,5.100,0.016,7.606,0.268,0.002,5.546,4,7
7,Ridge,A,5.100,0.016,7.606,0.268,0.002,5.546,5,8
0,"LightGBM cuantil 0,75",B,5.801,0.014,7.854,0.219,-2.236,6.725,6,1
1,"HistGradientBoosting cuantil 0,75",B,5.839,0.031,7.883,0.214,-2.278,6.768,7,2
4,"XGBoost cuantil 0,75",B,5.931,0.020,7.944,0.201,-2.334,6.787,8,5
9,Línea base: siempre la mediana,base,6.036,0.025,9.173,-0.065,2.262,6.007,9,10
8,Línea base: plazo fijo (percentil 75),base,7.535,0.039,9.459,-0.132,-3.231,8.829,10,9


## 5. Comparación por pares contra el mejor modelo

Pliegue por pliegue: ¿en cuántos gana el mejor y cuánto vale la diferencia frente a su ruido? Es **consistente** si gana en al menos
4 de 5 y la diferencia supera 1 vez su ruido; si no, es un **empate práctico** y gana el más simple y rápido.

In [7]:
best = table.loc[table["mejor"], "modelo"].iloc[0]
print("mejor modelo por costo (sin contar las líneas base):", best)
table.loc[~table["mejor"], ["modelo", "gana_vs_mejor", "ratio_vs_mejor", "consistente_vs_mejor", "empate_con_el_mejor"]].round(2)

mejor modelo por costo (sin contar las líneas base): LightGBM cuantil 0,75


,modelo,gana_vs_mejor,ratio_vs_mejor,consistente_vs_mejor,empate_con_el_mejor
1,"HistGradientBoosting cuantil 0,75",5 de 5,2.06,True,False
2,LightGBM,5 de 5,6.00,True,False
3,XGBoost,5 de 5,6.01,True,False
4,"XGBoost cuantil 0,75",5 de 5,6.84,True,False
5,Bosque aleatorio liviano,5 de 5,8.17,True,False
6,Regresión lineal,5 de 5,20.37,True,False
7,Ridge,5 de 5,20.33,True,False
8,Línea base: plazo fijo (percentil 75),5 de 5,39.47,True,False
9,Línea base: siempre la mediana,5 de 5,95.66,True,False


## 6. La regresión lineal contra el ganador

La regresión lineal es el modelo del caso de negocio y siempre compite. La diferencia se expresa en **unidades de costo** por pedido y por cada 1.000 pedidos
(1 unidad de costo = un día de sobrestimación). **No se le asigna un valor en dinero.** El MAE de la familia A y el de la B no se comparan entre sí.

In [8]:
LINEAR = "Regresión lineal"
costs = by_fold.pivot(index="pliegue", columns="modelo", values="costo")
versus = compare_models(costs[best].to_numpy(), costs[LINEAR].to_numpy())
row = table.set_index("modelo")
cost_gap = row.loc[LINEAR, "costo_media"] - row.loc[best, "costo_media"]
print(f"{LINEAR} frente a {best}:")
print(f"  costo: {cost_gap:+.3f} por pedido  ->  {per_thousand(cost_gap):+.0f} unidades de costo por cada 1.000 pedidos")
print(f"  el mejor gana en {versus['texto']} pliegues · diferencia / ruido = {versus['ratio']:.2f} · consistente: {versus['consistente']}")


Regresión lineal frente a LightGBM cuantil 0,75:
  costo: +0.298 por pedido  ->  +298 unidades de costo por cada 1.000 pedidos
  el mejor gana en 5 de 5 pliegues · diferencia / ruido = 20.37 · consistente: True


## 7. Revisión por región del cliente

Para **todos los modelos** (no solo los de arriba): MAE y porcentaje de pedidos que llega tarde por región, con predicciones
**fuera de muestra** (principio VIII: dónde rinde peor). El MAE solo se compara dentro de cada familia.

In [9]:
region_models = [m for m in table["modelo"] if m not in BASELINES.values()]
all_candidates = candidates()
regions = pd.concat([regional_metrics(m, all_candidates[m], X, y, folds) for m in region_models], ignore_index=True)
regions.pivot(index="grupo", columns="modelo", values="tarde").round(3)

modelo,Bosque aleatorio liviano,"HistGradientBoosting cuantil 0,75",LightGBM,"LightGBM cuantil 0,75",Regresión lineal,Ridge,XGBoost,"XGBoost cuantil 0,75"
grupo,,,,,,,,
Centro-Oeste,0.268,0.253,0.270,0.256,0.272,0.272,0.272,0.253
Nordeste,0.311,0.265,0.306,0.264,0.303,0.303,0.310,0.278
Norte,0.339,0.271,0.333,0.264,0.314,0.315,0.321,0.295
Sudeste,0.232,0.253,0.231,0.255,0.231,0.231,0.230,0.246
Sul,0.265,0.251,0.261,0.257,0.265,0.265,0.265,0.252


In [10]:
regions.pivot(index="grupo", columns="modelo", values="mae").round(3)

modelo,Bosque aleatorio liviano,"HistGradientBoosting cuantil 0,75",LightGBM,"LightGBM cuantil 0,75",Regresión lineal,Ridge,XGBoost,"XGBoost cuantil 0,75"
grupo,,,,,,,,
Centro-Oeste,5.420,6.278,5.237,6.200,5.339,5.339,5.286,6.390
Nordeste,7.368,8.634,7.305,8.606,7.339,7.339,7.325,8.463
Norte,7.741,9.014,7.765,9.043,7.756,7.756,7.786,8.719
Sudeste,4.548,5.239,4.494,5.207,4.625,4.625,4.497,5.365
Sul,5.447,6.269,5.306,6.203,5.447,5.447,5.323,6.413


## 8. ¿Se mantiene la calibración en el tiempo? Cobertura mes a mes hacia adelante

En los pliegues los plazos cubren ≈ 75 % de los pedidos. En la prueba hacia adelante (marzo a agosto de 2018), ¿cubren lo mismo cada mes?
Un plazo aprendido del pasado puede quedar corto en unos meses y largo en otros si cambia el ritmo de entrega.

In [11]:
monthly_models = [best, "HistGradientBoosting cuantil 0,75", "LightGBM", LINEAR]
monthly = pd.concat([coverage_by_month(m, all_candidates[m], fwd_train, fwd_valid) for m in monthly_models], ignore_index=True)
monthly.pivot(index="mes", columns="modelo", values="cobertura").round(3)

modelo,"HistGradientBoosting cuantil 0,75",LightGBM,"LightGBM cuantil 0,75",Regresión lineal
mes,,,,
2018-03,0.612,0.615,0.611,0.617
2018-04,0.831,0.830,0.829,0.822
2018-05,0.775,0.774,0.771,0.775
2018-06,0.922,0.925,0.922,0.925
2018-07,0.911,0.914,0.910,0.915
2018-08,0.933,0.941,0.931,0.944


In [12]:
# Qué pasó con los días reales de entrega mes a mes en la validación (iguales para todos los modelos)
monthly[monthly["modelo"] == best][["mes", "n_pedidos", "mediana_dias", "p75_dias"]].round(2)

,mes,n_pedidos,mediana_dias,p75_dias
0,2018-03,5650,13.40,21.57
1,2018-04,5490,9.37,14.40
2,2018-05,5467,9.10,15.69
3,2018-06,4879,7.96,11.55
4,2018-07,4892,7.53,10.93
5,2018-08,5069,7.01,9.80


## 9. Guardar las tablas

`reports/05_torneo_por_pliegue.csv` (cada modelo en cada pliegue y en la partición hacia adelante), `05_torneo_resumen.csv`,
`05_torneo_regiones.csv`, `05_torneo_cobertura_mensual.csv` y `05_torneo_tiempos.csv` (el tiempo de ajuste cambia entre ejecuciones y va aparte). El texto con el
motivo de cada decisión está en `reports/05_torneo.md`.

In [13]:
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
per_fold = pd.concat([by_fold.drop(columns="fit_seconds"), forward.drop(columns="fit_seconds")], ignore_index=True)
times = pd.concat([by_fold[["modelo", "pliegue", "fit_seconds"]], forward[["modelo", "pliegue", "fit_seconds"]]], ignore_index=True)

per_fold.round(6).to_csv(REPORTS_DIR / "05_torneo_por_pliegue.csv", index=False)
table.round(6).to_csv(REPORTS_DIR / "05_torneo_resumen.csv", index=False)
regions.round(6).to_csv(REPORTS_DIR / "05_torneo_regiones.csv", index=False)
monthly.round(6).to_csv(REPORTS_DIR / "05_torneo_cobertura_mensual.csv", index=False)
times.round(3).to_csv(REPORTS_DIR / "05_torneo_tiempos.csv", index=False)
print(f"guardado: {len(per_fold)} filas por pliegue (10 candidatos x 6) · {len(table)} en el resumen · {len(regions)} en regiones · {len(monthly)} en cobertura mensual")

guardado: 60 filas por pliegue (10 candidatos x 6) · 10 en el resumen · 40 en regiones · 24 en cobertura mensual


## 10. Las pruebas automáticas

Desde la raíz del repo:

```bash
pytest tests
```

Candidatos y modelo de plazo en `tests/test_models.py`; métricas en `tests/test_metrics.py`; evaluación, comparación y tabla del torneo en `tests/test_evaluation.py`.